In [ ]:
%load_ext rich
%load_ext autoreload
%autoreload 2

# End-to-End PDF Anonymization (PyMuPDF Layout + AymurAI API)
This notebook builds layout-based paragraphs from the source PDF, runs `/anonymizer/predict` + `/anonymizer/disambiguate`, and compiles an anonymized PDF.


In [ ]:
import json
import time
from pathlib import Path

import pymupdf
import requests
from tqdm.auto import tqdm

In [ ]:
# Change these values to test different documents/environments.
API_URL = "http://localhost:8999"
SOURCE_PDF = Path("./document.pdf")

OUTPUT_DIR = Path("./output")
USE_CACHE = False

# Optional: keep as None to rely on backend default policies.
LABEL_POLICIES = None

# Keep aligned with current anonymizer defaults.
RENDER_POLICY = {"suffix_mode": "auto", "suffix_threshold": 1}

SOURCE_PDF

In [ ]:
def extract_document_via_api(pdf_path: Path) -> dict:
    with pdf_path.open("rb") as handle:
        response = requests.post(
            f"{API_URL}/document-extract",
            files={"file": (pdf_path.name, handle, "application/pdf")},
            timeout=600,
        )

    response.raise_for_status()
    return response.json()


def predict_paragraph(text: str, retries: int = 2) -> dict:
    last_error = None
    for attempt in range(retries + 1):
        try:
            response = requests.post(
                f"{API_URL}/anonymizer/predict",
                json={"text": text},
                params={"use_cache": USE_CACHE},
                timeout=600,
            )
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            if attempt < retries:
                time.sleep(2)
            else:
                raise last_error

    raise RuntimeError("Predict request exhausted retries")


def disambiguate(predictions: list[dict]) -> dict:
    payload = {"paragraphs": predictions}
    if LABEL_POLICIES is not None:
        payload["label_policies"] = LABEL_POLICIES

    response = requests.post(
        f"{API_URL}/anonymizer/disambiguate",
        json=payload,
        timeout=600,
    )
    response.raise_for_status()
    return response.json()


def compile_pdf(pdf_path: Path, annotations: dict) -> Path:
    payload = {
        "data": annotations["data"],
        "render_policy": RENDER_POLICY,
    }
    if annotations.get("label_policies") is not None:
        payload["label_policies"] = annotations["label_policies"]

    with pdf_path.open("rb") as handle:
        response = requests.post(
            f"{API_URL}/anonymizer/anonymize-document",
            data={"annotations": json.dumps(payload, ensure_ascii=False)},
            files={"file": (pdf_path.name, handle, "application/pdf")},
            timeout=1200,
        )

    response.raise_for_status()

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIR / f"{pdf_path.stem}.anonymized.pdf"
    output_path.write_bytes(response.content)
    return output_path

In [ ]:
document_extract_payload = extract_document_via_api(SOURCE_PDF)
paragraphs = document_extract_payload["document"]

print(f"Document ID: {document_extract_payload['document_id']}")
print(f"Paragraphs extracted: {len(paragraphs)}")

paragraphs[:5]

In [ ]:
predictions = [
    predict_paragraph(paragraph)
    for paragraph in tqdm(paragraphs, desc="Predicting paragraphs")
]
total_labels = sum(len(pred.get("labels") or []) for pred in predictions)
print(f"Predictions: {len(predictions)} paragraphs, {total_labels} labels")

In [ ]:
disambiguated = disambiguate(predictions)
total_labels = sum(len(pred.get("labels") or []) for pred in disambiguated["data"])
print(f"Disambiguated labels: {total_labels}")
disambiguated.keys()

In [ ]:
[data for data in disambiguated["data"] if data["labels"]]

In [ ]:
output_pdf = compile_pdf(SOURCE_PDF, disambiguated)
print(output_pdf.resolve())
output_pdf

In [ ]:
with pymupdf.open(str(output_pdf)) as doc:
    watermark_hits = sum(
        len(page.search_for("Documento anonimizado por AymurAI")) for page in doc
    )
    print(f"Pages: {doc.page_count}")
    print(f"Watermark hits: {watermark_hits}")